In [2]:
!pip install -q sentencepiece sacrebleu wandb pyyaml tqdm nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.1 MB/s eta 0:00:00


In [1]:
!git clone https://github.com/Annapoorneshwari-Murthy/cw2_2025_2026-students.git

Cloning into 'cw2_2025_2026-students'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 106 (delta 26), reused 16 (delta 16), pack-reused 65 (from 1)
Receiving objects: 100% (106/106), 33.69 MiB | 24.94 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [3]:
!ls -la

total 20
drwxr-xr-x 1 root root 4096 Nov 25 23:52 .
drwxr-xr-x 1 root root 4096 Nov 25 23:50 ..
drwxr-xr-x 4 root root 4096 Nov 20 14:30 .config
drwxr-xr-x 8 root root 4096 Nov 25 23:52 cw2_2025_2026-students
drwxr-xr-x 1 root root 4096 Nov 20 14:30 sample_data


In [4]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sannapoorna15 (sannapoorna15-heriot-watt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
os.chdir("cw2_2025_2026-students/part1")
print(f"Current directory: {os.getcwd()}")
!ls -la

Current directory: /content/cw2_2025_2026-students/part1
total 116
drwxr-xr-x 3 root root  4096 Nov 25 23:52 .
drwxr-xr-x 8 root root  4096 Nov 25 23:52 ..
-rw-r--r-- 1 root root   676 Nov 25 23:52 evaluate_all_models.ps1
-rw-r--r-- 1 root root  4103 Nov 25 23:52 INSTRUCTIONS.md
-rw-r--r-- 1 root root   156 Nov 25 23:52 model_config_large.yaml
-rw-r--r-- 1 root root   154 Nov 25 23:52 model_config_medium.yaml
-rw-r--r-- 1 root root   154 Nov 25 23:52 model_config_small.yaml
-rw-r--r-- 1 root root   152 Nov 25 23:52 model_config.yaml
-rw-r--r-- 1 root root 18350 Nov 25 23:52 nmt_model.py
-rw-r--r-- 1 root root  2973 Nov 25 23:52 predict.py
-rw-r--r-- 1 root root  2383 Nov 25 23:52 sanity_check.pt
-rw-r--r-- 1 root root  2334 Nov 25 23:52 sanity_check.py
-rw-r--r-- 1 root root  4790 Nov 25 23:52 test.py
-rw-r--r-- 1 root root  1053 Nov 25 23:52 train_all_models.ps1
-rw-r--r-- 1 root root   441 Nov 25 23:52 train_config.yaml
-rw-r--r-- 1 root root 10984 Nov 25 23:52 train.py
-rw-r--r-- 1 

In [6]:
!python vocab.py

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
read in source sentences: ../multi30k_data/train.en
read in target sentences: ../multi30k_data/train.fr
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ../multi30k_data/train.en
  input_format: 
  model_prefix: src
  model_type: UNIGRAM
  vocab_size: 7000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_fil

In [9]:
import re

# Read the nmt_model.py file
with open('nmt_model.py', 'r') as f:
    content = f.read()

# Check if implementation already exists
if "e_t = torch.bmm" in content and "alpha_t = F.softmax" in content:
    print("Dot-product attention is already implemented!")
else:
    print("Adding dot-product attention implementation...")

    # The complete implementation code
    implementation_code = """        ### Your code here (~8-15 lines) ###
        # Dot-product attention
        # 2. Compute attention scores e_t
        # Need to compute batched matrix multiplication between dec_hidden and enc_hiddens_proj
        # dec_hidden has a shape of (b, h), enc_hiddens_proj is (b, src_len, h)
        # We want to end up with a shape of (b, src_len)
        # e_t = dec_hidden @ enc_hiddens_proj^T
        e_t = torch.bmm(dec_hidden.unsqueeze(1), enc_hiddens_proj.transpose(1, 2)).squeeze(1)  # (b, src_len)

        # If enc_masks is None, this step should be skipped
        # Use bool() to convert ByteTensor to BoolTensor
        # Use float("-inf") to represent -inf
        # Use masked_fill_ to fill in -inf at the masked positions
        if enc_masks is not None:
            e_t = e_t.masked_fill(enc_masks.bool(), float("-inf"))

        # 3. Apply softmax to e_t to yield alpha_t of shape (b, src_len)
        alpha_t = F.softmax(e_t, dim=1)  # (b, src_len)

        # 4. Use batched matrix multiplication between alpha_t and enc_hiddens
        # alpha_t has a shape of (b, src_len), enc_hiddens is (b, src_len, 2h)
        # We want to end up with a shape of (b, 2h)
        attention_t = torch.bmm(alpha_t.unsqueeze(1), enc_hiddens).squeeze(1)  # (b, 2h)

        # 5. Concatenate dec_hidden with attention_t to compute tensor u_t
        u_t = torch.cat((dec_hidden, attention_t), dim=1)  # (b, 3h)

        # 6. Apply combined output projection layer to u_t to compute tensor v_t
        v_t = self.combined_output_projection(u_t)  # (b, h)

        # 7. Compute tensor O_t by applying Tanh and then dropout to v_t
        o_t = self.dropout(torch.tanh(v_t))  # (b, h)

        ### End of your code ###"""

    # Find the placeholder and replace it
    pattern = r'### Your code here.*?### End of your code ###'

    if re.search(pattern, content, re.DOTALL):
        content = re.sub(pattern, implementation_code, content, flags=re.DOTALL)

        # Write back to file
        with open('nmt_model.py', 'w') as f:
            f.write(content)

        print("Dot-product attention implementation added successfully!")
        print("You can now run sanity_check.py")
    else:
        print("Could not find the code placeholder.")
        print("Please check that nmt_model.py has the '### Your code here' section.")

Dot-product attention is already implemented!


In [10]:
!python sanity_check.py

All checks passed!


In [11]:
!python train.py \
    --model-config model_config_small.yaml \
    --train-config train_config.yaml \
    --checkpoint-path models/small \
    --wandb-project nmt-project \
    --wandb-run-name bilstm-small
print("Small model training complete!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Train dataset size: 29000
Dev dataset size: 1014
Load vocab from vocab/vocab.json
uniformly initialize parameters [-0.1, +0.1]
use device: cuda
Starting Training!
wandb: Currently logged in as: sannapoorna15 (sannapoorna15-heriot-watt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.23.0
wandb: Run data is saved locally in /content/cw2_2025_2026-students/part1/wandb/run-20251125_235720-zrnttidv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm-small
wandb: ⭐️ View project at https://wandb.ai/sannapoorna15-heriot-watt-university/nmt-project
wandb: 🚀 View run at https://wandb.ai/sannapoorna15-heriot-watt-university/nmt-project/runs/zrnttidv
1
epoch 1, iter 10, avg. 

In [13]:
!python train.py \
    --model-config model_config_medium.yaml \
    --train-config train_config.yaml \
    --checkpoint-path models/medium \
    --wandb-project nmt-project \
    --wandb-run-name bilstm-medium
print("Medium model training complete!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Train dataset size: 29000
Dev dataset size: 1014
Load vocab from vocab/vocab.json
uniformly initialize parameters [-0.1, +0.1]
use device: cuda
Starting Training!
wandb: Currently logged in as: sannapoorna15 (sannapoorna15-heriot-watt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.23.0
wandb: Run data is saved locally in /content/cw2_2025_2026-students/part1/wandb/run-20251126_002027-ivf6axf4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm-medium
wandb: ⭐️ View project at https://wandb.ai/sannapoorna15-heriot-watt-university/nmt-project
wandb: 🚀 View run at https://wandb.ai/sannapoorna15-heriot-watt-university/nmt-project/runs/ivf6axf4
1
epoch 1, iter 10, avg.

In [14]:
!python train.py \
    --model-config model_config_large.yaml \
    --train-config train_config.yaml \
    --checkpoint-path models/large \
    --wandb-project nmt-project \
    --wandb-run-name bilstm-large
print("Large model training complete!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Train dataset size: 29000
Dev dataset size: 1014
Load vocab from vocab/vocab.json
uniformly initialize parameters [-0.1, +0.1]
use device: cuda
Starting Training!
wandb: Currently logged in as: sannapoorna15 (sannapoorna15-heriot-watt-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.23.0
wandb: Run data is saved locally in /content/cw2_2025_2026-students/part1/wandb/run-20251126_004312-7uu7bwdb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm-large
wandb: ⭐️ View project at https://wandb.ai/sannapoorna15-heriot-watt-university/nmt-project
wandb: 🚀 View run at https://wandb.ai/sannapoorna15-heriot-watt-university/nmt-project/runs/7uu7bwdb
1
epoch 1, iter 10, avg. loss 8.75 cum. examples 320, speed 22

In [16]:
# Test Small Model
print("Testing Small Model...")
!python test.py \
    --checkpoint-path models/small/nmt.model \
    --output-file outputs/small_results.json \
    --vocab-file vocab/vocab.json

# Test Medium Model
print("\nTesting Medium Model...")
!python test.py \
    --checkpoint-path models/medium/nmt.model \
    --output-file outputs/medium_results.json \
    --vocab-file vocab/vocab.json

# Test Large Model
print("\nTesting Large Model...")
!python test.py \
    --checkpoint-path models/large/nmt.model \
    --output-file outputs/large_results.json \
    --vocab-file vocab/vocab.json

print("\nAll Part 1 models tested!")

Testing Small Model...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Load test source sentences from ../multi30k_data/test_2016_flickr.en
Load test target sentences from ../multi30k_data/test_2016_flickr.fr
Load vocab from vocab/vocab.json
Load model from models/small/nmt.model
Generating Predictions: 100% 1000/1000 [00:08<00:00, 114.55it/s]
Corpus BLEU score = 23.16
Saving predictions to outputs/small_results.json

Testing Medium Model...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Load test source sentences from ../multi30k_data/test_2016_flickr.en
Load test target sentences from ../multi30k_data/test_2016_flickr.fr
Load vocab from vocab/vocab.json
Load model from models/medium/nmt.model
Generating Predictions: 100% 1000/1000 [00:08<00:00, 120.97it/s]
Corpus BLEU score = 47.37
Saving predictions to outputs/medium_results.json

Testing Lar

In [17]:
import json

print("\n" + "="*60)
print("PART I: Bi-LSTM Results")
print("="*60)

results = {}
for model_size in ['small', 'medium', 'large']:
    try:
        with open(f'outputs/{model_size}_results.json', 'r') as f:
            data = json.load(f)
            results[model_size] = data['bleu_score']
            print(f"{model_size.capitalize():8s}: BLEU = {data['bleu_score']:.2f}")
    except FileNotFoundError:
        results[model_size] = 'Not evaluated'
        print(f"{model_size.capitalize():8s}: Not evaluated")

print("="*60)


PART I: Bi-LSTM Results
Small   : BLEU = 23.16
Medium  : BLEU = 47.37
Large   : BLEU = 52.37


Part - II

In [18]:
import os
os.chdir("../part2")
print(f"Current directory: {os.getcwd()}")
!ls -la


Current directory: /content/cw2_2025_2026-students/part2
total 116
drwxr-xr-x 3 root root  4096 Nov 25 23:52 .
drwxr-xr-x 8 root root  4096 Nov 25 23:52 ..
-rw-r--r-- 1 root root   596 Nov 25 23:52 evaluate_models.ps1
-rw-r--r-- 1 root root   188 Nov 25 23:52 model_config_large.yaml
-rw-r--r-- 1 root root   187 Nov 25 23:52 model_config_medium.yaml
-rw-r--r-- 1 root root   187 Nov 25 23:52 model_config_small.yaml
-rw-r--r-- 1 root root   188 Nov 25 23:52 model_config.yaml
-rw-r--r-- 1 root root 13997 Nov 25 23:52 model.py
-rw-r--r-- 1 root root  6053 Nov 25 23:52 predict.py
-rw-r--r-- 1 root root  2380 Nov 25 23:52 sanity_check.pt
-rw-r--r-- 1 root root  1095 Nov 25 23:52 sanity_check.py
-rw-r--r-- 1 root root  6072 Nov 25 23:52 test.py
-rw-r--r-- 1 root root   642 Nov 25 23:52 train_config.yaml
-rw-r--r-- 1 root root  3037 Nov 25 23:52 TRAINING_STATUS.md
-rw-r--r-- 1 root root 12638 Nov 25 23:52 train.py
-rw-r--r-- 1 root root  3249 Nov 25 23:52 utils.py
drwxr-xr-x 2 root root  4096 N

In [19]:
!python vocab.py --source-vocab-size 14563
print("Vocabulary built for Part 2!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: vocab/spm
  model_type: UNIGRAM
  vocab_size: 14563
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <

In [20]:
import re

print("="*70)
print("Checking Part II Implementation Status")
print("="*70)

with open('model.py', 'r') as f:
    content = f.read()

# Check key components
checks = {
    "Query computation": "q = self.query(x)" in content,
    "Key computation": "k = self.key(x)" in content,
    "Value computation": "v = self.value(x)" in content,
    "Attention scores": "q @ k.transpose" in content,
    "Causal masking": "masked_fill" in content and "mask" in content,
    "Softmax": "F.softmax(att" in content,
    "Apply to values": "att @ v" in content,
    "Output projection": "self.proj(y)" in content,
}

print("\n Implementation Checks:")
for check_name, passed in checks.items():
    status = "checked" if passed else "not checked"
    print(f"{status} {check_name}")

# Check for placeholder
has_placeholder = "### Your code here" in content

print("\n" + "="*70)
if all(checks.values()) and not has_placeholder:
    print("RESULT: Implementation is COMPLETE!")
    print("   Run: !python sanity_check.py to verify")
else:
    print("RESULT: Implementation needs attention.")
    if has_placeholder:
        print("   Placeholder found - run implementation cell")
print("="*70)

Checking Part II Implementation Status

 Implementation Checks:
checked Query computation
checked Key computation
checked Value computation
checked Attention scores
checked Causal masking
checked Softmax
checked Apply to values
checked Output projection

RESULT: Implementation needs attention.
   Placeholder found - run implementation cell


In [21]:
!python sanity_check.py
print("Sanity check complete!")

Number of parameters: 6240 (0.01M)
All checks passed!
Sanity check complete!


In [22]:
!python train.py \
    --model-config model_config_small.yaml \
    --train-config train_config.yaml \
    --checkpoint-path models/small \
    --wandb-project nmt-transformer \
    --wandb-run-name transformer-small
print("Small transformer training complete!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Using device: cuda
Number of parameters: 474752 (0.47M)
GPT(
  (tok_emb): Embedding(14566, 16)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (key): Linear(in_features=16, out_features=16, bias=True)
        (query): Linear(in_features=16, out_features=16, bias=True)
        (value): Linear(in_features=16, out_features=16, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
        (proj): Linear(in_features=16, out_features=16, bias=True)
      )
      (mlp): Sequential(
        (0): Linear(in_features=16, out_features=64, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=64, out_features=16, bias=True)
        

In [23]:
!python train.py \
    --model-config model_config_medium.yaml \
    --train-config train_config.yaml \
    --checkpoint-path models/medium \
    --wandb-project nmt-transformer \
    --wandb-run-name transformer-medium
print("Medium transformer training complete!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Using device: cuda
Number of parameters: 2072704 (2.07M)
GPT(
  (tok_emb): Embedding(14566, 64)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
        (proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (mlp): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
     

In [24]:
!python train.py \
    --model-config model_config_large.yaml \
    --train-config train_config.yaml \
    --checkpoint-path models/large \
    --wandb-project nmt-transformer \
    --wandb-run-name transformer-large
print("Large transformer training complete!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Using device: cuda
Number of parameters: 8287488 (8.29M)
GPT(
  (tok_emb): Embedding(14566, 192)
  (blocks): Sequential(
    (0): Block(
      (ln1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (key): Linear(in_features=192, out_features=192, bias=True)
        (query): Linear(in_features=192, out_features=192, bias=True)
        (value): Linear(in_features=192, out_features=192, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
        (proj): Linear(in_features=192, out_features=192, bias=True)
      )
      (mlp): Sequential(
        (0): Linear(in_features=192, out_features=768, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=768, out_features=192, bia

In [25]:
# Test Small Model
print("Testing Small Transformer Model...")
!python test.py \
    --checkpoint-path models/small/nmt.model \
    --model-config model_config_small.yaml \
    --output-file outputs/small_results.json

# Test Medium Model
print("\nTesting Medium Transformer Model...")
!python test.py \
    --checkpoint-path models/medium/nmt.model \
    --model-config model_config_medium.yaml \
    --output-file outputs/medium_results.json

# Test Large Model
print("\nTesting Large Transformer Model...")
!python test.py \
    --checkpoint-path models/large/nmt.model \
    --model-config model_config_large.yaml \
    --output-file outputs/large_results.json

print("\n All Part 2 models tested!")


Testing Small Transformer Model...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Load test source sentences from ../multi30k_data/test_2016_flickr.en
Load test target sentences from ../multi30k_data/test_2016_flickr.fr
Load vocab from vocab/vocab.json
Load model from models/small/nmt.model
Number of parameters: 474752 (0.47M)
Generating Predictions: 100% 1000/1000 [00:24<00:00, 40.84it/s]
Corpus BLEU score = 20.03
Saving predictions to outputs/small_results.json

Testing Medium Transformer Model...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Load test source sentences from ../multi30k_data/test_2016_flickr.en
Load test target sentences from ../multi30k_data/test_2016_flickr.fr
Load vocab from vocab/vocab.json
Load model from models/medium/nmt.model
Number of parameters: 2072704 (2.07M)
Generating Predictions: 100% 1000/1000 [00:46<00:00, 21

In [26]:
import json

print("\n" + "="*60)
print("PART II: Transformer Results")
print("="*60)

results = {}
for model_size in ['small', 'medium', 'large']:
    try:
        with open(f'outputs/{model_size}_results.json', 'r') as f:
            data = json.load(f)
            results[model_size] = data['bleu_score']
            print(f"{model_size.capitalize():8s}: BLEU = {data['bleu_score']:.2f}")
    except FileNotFoundError:
        results[model_size] = 'Not evaluated'
        print(f"{model_size.capitalize():8s}: Not evaluated")

print("="*60)


PART II: Transformer Results
Small   : BLEU = 20.03
Medium  : BLEU = 46.95
Large   : BLEU = 50.40


Part III

In [27]:
import os
os.chdir("../part3")

# List all heatmap images
print("Available heatmaps:")
!ls *.png

# Display heatmaps (optional - you can view them in file browser)
from IPython.display import Image, display

print("\n Heatmaps are in the part3 folder.")
print(" Analyze them and answer question g on Canvas.")
print(" Compare Bi-LSTM vs Transformer outputs for the 5 example sentences.")

Available heatmaps:
bilstm_example1.png  bilstm_example5.png  trans_example4.png
bilstm_example2.png  trans_example1.png   trans_example5.png
bilstm_example3.png  trans_example2.png
bilstm_example4.png  trans_example3.png

 Heatmaps are in the part3 folder.
 Analyze them and answer question g on Canvas.
 Compare Bi-LSTM vs Transformer outputs for the 5 example sentences.
